# K-Means聚类 + Monte Carlo 组合优化

**参考论文**: Ziwei Wang (2022), "300 Stocks Based on Monte Carlo Algorithm"  
**核心思想**: 先用K-Means将ETF按收益/风险特征聚类，再在每个簇内/簇间用Monte Carlo模拟筛选有效前沿组合  
**方法**: 聚类降维 + 随机模拟 + 有效前沿提取

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 策略原理

### Step 1: K-Means 聚类

将 $N$ 个资产按 (年化收益, 年化波动率) 特征向量聚为 $K$ 类:

$$\min_{C_1,\ldots,C_K} \sum_{k=1}^{K} \sum_{i \in C_k} \|x_i - \mu_k\|^2$$

其中 $x_i = (\bar{r}_i, \sigma_i)$，$\mu_k$ 为第 $k$ 簇中心。

### Step 2: Monte Carlo 随机组合生成

生成 $M$ 个随机权重组合 (Dirichlet分布):
$$w^{(m)} \sim \text{Dirichlet}(\alpha_1, \ldots, \alpha_N), \quad m = 1,\ldots,M$$

计算每个组合的收益与风险:
$$R_p^{(m)} = w^{(m)T} \mu, \quad \sigma_p^{(m)} = \sqrt{w^{(m)T} \Sigma w^{(m)}}$$

### Step 3: 有效前沿提取

从 $M$ 个组合中筛选Pareto最优:
$$\mathcal{F} = \{m : \nexists m' \text{ s.t. } R_p^{(m')} \geq R_p^{(m)} \text{ and } \sigma_p^{(m')} \leq \sigma_p^{(m)}\}$$

### Step 4: 最优组合选择

在有效前沿上选择最大夏普比率组合:
$$w^* = \arg\max_{m \in \mathcal{F}} \frac{R_p^{(m)} - r_f}{\sigma_p^{(m)}}$$

In [ ]:
# Cell 3: 动画 - Monte Carlo 散点逐步填充有效前沿
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

np.random.seed(42)

# 使用shared的动画函数 (先用模拟数据演示)
from shared.animation_helpers import animate_efficient_frontier

# 模拟8个资产的收益矩阵 (500天)
n_days, n_assets = 500, 8
sim_returns = np.random.multivariate_normal(
    mean=np.array([0.0003, 0.0002, 0.0005, 0.0004, 0.0006, 0.0007, 0.0005, 0.0001]),
    cov=np.eye(n_assets) * 0.0003 + np.ones((n_assets, n_assets)) * 0.0001,
    size=n_days
)
sim_returns_df = pd.DataFrame(sim_returns, columns=[f'Asset_{i}' for i in range(n_assets)])

fig = animate_efficient_frontier(sim_returns_df, title='Monte Carlo: 有效前沿逐步生成')
fig.show()

In [ ]:
# Cell 4: 导入依赖
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from shared.data_fetcher import get_etf_daily, get_stock_daily, get_index_daily
from shared.constants import (SECTOR_ETFS, DEFAULT_START, DEFAULT_END,
                               RISK_FREE_RATE, TRADING_DAYS_PER_YEAR, INITIAL_CAPITAL)
from shared.metrics import summary_table
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)

np.random.seed(42)
print('All imports loaded successfully.')

In [ ]:
# Cell 5: 下载行业ETF数据
etf_names = list(SECTOR_ETFS.keys())
etf_codes = list(SECTOR_ETFS.values())

fallback_stocks = {
    '银行': '601398', '券商': '600030', '医药': '600276', '消费': '000858',
    '科技': '002415', '新能源': '300750', '军工': '600893', '地产': '001979',
}

close_dict = {}
for name, code in SECTOR_ETFS.items():
    df = get_etf_daily(code, DEFAULT_START, DEFAULT_END)
    if df is not None and len(df) > 100 and 'close' in df.columns:
        close_dict[name] = df['close']
    else:
        print(f'ETF {name}({code}) 失败, 使用备用个股')
        df = get_stock_daily(fallback_stocks[name], DEFAULT_START, DEFAULT_END)
        if df is not None and len(df) > 100 and 'close' in df.columns:
            close_dict[name] = df['close']

prices_df = pd.DataFrame(close_dict).sort_index().ffill().dropna()

bench_df = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
bench_close = bench_df['close'] if len(bench_df) > 0 else None

print(f'数据区间: {prices_df.index[0].date()} ~ {prices_df.index[-1].date()}')
print(f'资产数量: {prices_df.shape[1]}, 交易日: {prices_df.shape[0]}')

In [ ]:
# Cell 6: K-Means聚类 + Monte Carlo优化函数

def cluster_assets(returns_df, n_clusters=3):
    """
    K-Means聚类: 按 (年化收益, 年化波动率) 分组
    """
    annual_ret = returns_df.mean() * TRADING_DAYS_PER_YEAR
    annual_vol = returns_df.std() * np.sqrt(TRADING_DAYS_PER_YEAR)

    features = pd.DataFrame({
        'return': annual_ret,
        'volatility': annual_vol,
    })

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    features['cluster'] = labels
    return features, labels


def monte_carlo_portfolios(returns_df, n_portfolios=5000):
    """
    Monte Carlo随机生成组合, 返回风险/收益/权重
    """
    n_assets = returns_df.shape[1]
    mean_returns = returns_df.mean().values * TRADING_DAYS_PER_YEAR
    cov_matrix = returns_df.cov().values * TRADING_DAYS_PER_YEAR

    results = np.zeros((n_portfolios, 3))  # return, risk, sharpe
    all_weights = np.zeros((n_portfolios, n_assets))

    for i in range(n_portfolios):
        w = np.random.dirichlet(np.ones(n_assets))
        port_ret = w @ mean_returns
        port_risk = np.sqrt(w @ cov_matrix @ w)
        port_sharpe = (port_ret - RISK_FREE_RATE) / port_risk if port_risk > 0 else 0

        results[i] = [port_ret, port_risk, port_sharpe]
        all_weights[i] = w

    return results, all_weights


def extract_efficient_frontier(results, all_weights, n_bins=50):
    """
    从Monte Carlo结果中提取近似有效前沿
    在每个风险区间取最高收益的组合
    """
    risk_min, risk_max = results[:, 1].min(), results[:, 1].max()
    risk_bins = np.linspace(risk_min, risk_max, n_bins)

    frontier_idx = []
    for j in range(len(risk_bins) - 1):
        mask = (results[:, 1] >= risk_bins[j]) & (results[:, 1] < risk_bins[j+1])
        if mask.sum() > 0:
            best = np.argmax(results[mask, 0])
            actual_idx = np.where(mask)[0][best]
            frontier_idx.append(actual_idx)

    return frontier_idx


def select_optimal_portfolio(results, all_weights, method='max_sharpe'):
    """选择最优组合"""
    if method == 'max_sharpe':
        idx = np.argmax(results[:, 2])
    elif method == 'min_risk':
        idx = np.argmin(results[:, 1])
    else:
        idx = np.argmax(results[:, 2])
    return all_weights[idx], results[idx]


print('K-Means + Monte Carlo 函数定义完成')

In [ ]:
# Cell 7: 滚动窗口回测

returns_df = prices_df.pct_change().dropna()
asset_names = list(prices_df.columns)
n_assets = len(asset_names)

lookback = 120
rebalance_freq = 20
n_mc = 3000
n_clusters = 3

dates = returns_df.index

# MC+KMeans策略
port_ret_mc = pd.Series(0.0, index=dates)
weights_mc = np.ones(n_assets) / n_assets
mc_weight_hist = []
cluster_hist = []

# 等权基准
port_ret_ew = pd.Series(0.0, index=dates)
weights_ew = np.ones(n_assets) / n_assets

rebal_dates = []

for i in range(lookback, len(dates)):
    daily_ret = returns_df.iloc[i].values
    port_ret_mc.iloc[i] = weights_mc @ daily_ret
    port_ret_ew.iloc[i] = weights_ew @ daily_ret

    if (i - lookback) % rebalance_freq == 0:
        window_ret = returns_df.iloc[i-lookback:i]

        # K-Means聚类
        cluster_info, labels = cluster_assets(window_ret, n_clusters)
        cluster_hist.append(cluster_info)

        # Monte Carlo随机组合
        mc_results, mc_weights = monte_carlo_portfolios(window_ret, n_mc)

        # 选择最大夏普比率组合
        opt_w, opt_stats = select_optimal_portfolio(mc_results, mc_weights, 'max_sharpe')
        weights_mc = opt_w

        # 交易成本
        if len(rebal_dates) > 0:
            port_ret_mc.iloc[i] -= 0.002

        rebal_dates.append(dates[i])
        mc_weight_hist.append(weights_mc.copy())

# 截取
port_ret_mc = port_ret_mc.iloc[lookback:]
port_ret_ew = port_ret_ew.iloc[lookback:]
equity_mc = INITIAL_CAPITAL * (1 + port_ret_mc).cumprod()
equity_ew = INITIAL_CAPITAL * (1 + port_ret_ew).cumprod()

print(f'调仓次数: {len(rebal_dates)}')
print(f'MC+KMeans终端净值: {equity_mc.iloc[-1]:,.0f}')
print(f'等权终端净值: {equity_ew.iloc[-1]:,.0f}')

In [ ]:
# Cell 8: 回测统计 + 聚类分析

bench_returns = None
if bench_close is not None:
    bench_close_aligned = bench_close.reindex(port_ret_mc.index).ffill()
    bench_returns = bench_close_aligned.pct_change().fillna(0)

metrics_mc = summary_table(port_ret_mc, equity_mc, bench_returns)
metrics_ew = summary_table(port_ret_ew, equity_ew, bench_returns)

compare_keys = ['年化收益率', '年化波动率', '夏普比率', '最大回撤', 'Calmar比率']
print(f'{"指标":<12} {"MC+KMeans":<14} {"等权":<14}')
print('-' * 42)
for k in compare_keys:
    print(f'{k:<12} {metrics_mc.get(k, "N/A"):<14} {metrics_ew.get(k, "N/A"):<14}')

# 最新聚类结果
if cluster_hist:
    latest_cluster = cluster_hist[-1]
    print('\n=== 最新聚类结果 ===')
    for c in range(n_clusters):
        members = latest_cluster[latest_cluster['cluster'] == c].index.tolist()
        print(f'Cluster {c}: {members}')

# 最新权重
if mc_weight_hist:
    print('\n=== 最新MC优化权重 ===')
    for name, w in zip(asset_names, mc_weight_hist[-1]):
        if w > 0.01:
            print(f'  {name}: {w:.1%}')

In [ ]:
# Cell 9: 可视化
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 1) 策略对比
plot_cumulative_comparison(
    {'MC+KMeans': port_ret_mc, '等权基准': port_ret_ew},
    title='Monte Carlo + K-Means vs 等权基准'
)

# 2) 回撤
plot_drawdown(equity_mc, title='MC+KMeans策略 - 回撤')

# 3) 聚类散点图 (最新一期)
if cluster_hist:
    fig, ax = plt.subplots(figsize=(10, 7))
    latest = cluster_hist[-1]
    colors_cluster = ['#2196F3', '#F44336', '#4CAF50']
    for c in range(n_clusters):
        mask = latest['cluster'] == c
        ax.scatter(latest.loc[mask, 'volatility'], latest.loc[mask, 'return'],
                   c=colors_cluster[c], s=120, label=f'Cluster {c}',
                   edgecolors='black', linewidth=0.5, zorder=5)
        for name in latest.loc[mask].index:
            ax.annotate(name, (latest.loc[name, 'volatility'], latest.loc[name, 'return']),
                        fontsize=10, ha='left', va='bottom')
    ax.set_title('K-Means 资产聚类 (收益-风险空间)', fontsize=14, fontweight='bold')
    ax.set_xlabel('年化波动率')
    ax.set_ylabel('年化收益率')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 4) 权重堆积图
weight_df = pd.DataFrame(mc_weight_hist, columns=asset_names,
                          index=rebal_dates[:len(mc_weight_hist)])
fig, ax = plt.subplots(figsize=(14, 5))
weight_df.plot.area(ax=ax, alpha=0.75, colormap='Set2')
ax.set_title('MC+KMeans - 资产权重演变', fontsize=14, fontweight='bold')
ax.set_ylabel('权重')
ax.set_ylim(0, 1)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
plt.tight_layout()
plt.show()

# 5) 绩效表
plot_metrics_table(metrics_mc, title='MC+KMeans 策略绩效指标')

## 结果分析

### 方法特点
1. **K-Means聚类**将资产按风险-收益特征自然分组，有助于理解资产间的相似性
2. **Monte Carlo模拟**无需求解优化问题，通过大量随机抽样覆盖权重空间
3. 有效前沿的提取直观地展示了风险-收益权衡

### 优缺点
- 优点: 不需要协方差矩阵可逆，对约束条件灵活，易于并行化
- 缺点: 采样效率低（高维空间需要大量样本），计算成本高
- 聚类数K的选择需要依据Elbow法则或轮廓系数

### 改进方向
- 使用Quasi-Monte Carlo (Sobol序列) 提高采样效率
- 加入Bayes优化替代纯随机搜索
- 聚类特征可扩展为多维 (加入偏度、峰度、相关性等)